In [ ]:
#Import required libraries for the maths, graphs, filter,interactable slider
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from IPython.display import display, Markdown

In [ ]:
# Generating the clean signal
duration = 1.0
sample_rate = 500
step = 5
frequency = 5

t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
clean_signal = np.sin(2 * np.pi * frequency * t)

# Adding Gaussian noise to a signal based on target SNR in dB
def add_noise(sig, snr_db):
    power = np.mean(sig**2)
    noise_power = power / (10 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), len(sig))
    return sig + noise

# Trying to recover the original signal by filtering the noise by using a Butterworth low-pass filter
def recover_signal(noisy, cutoff=20, fs=1000, order=4):
    b, a = signal.butter(order, cutoff / (fs / 2), btype='low')
    return signal.filtfilt(b, a, noisy)

# Measure recovered SNR by comparing estimate to original.
def measure_snr(original, estimate):
    noise = original - estimate
    signal_power = np.mean(original**2)
    noise_power = np.mean(noise**2)
    return 10 * np.log10(signal_power / noise_power)

# Map SNR to a real world scenario
def get_scenario_label(snr_db):
    if snr_db >= 18:
        return "Clear link / low interference"
    elif snr_db >= 8:
        return "Moderate interference / congested channel"
    else:
        return "Heavy degradation / severe channel noise"

# Plot magnitude spectrum of a signal.
def plot_spectrum(ax, sig, fs, title):
    n = len(sig)
    freqs = np.fft.rfftfreq(n, d=1/fs)
    spectrum = np.abs(np.fft.rfft(sig))
    ax.plot(freqs, spectrum)
    ax.set_title(title)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Magnitude")
    ax.grid(alpha=0.3)


# Interactive simulation for signal corruption and recovery
def simulate(snr_db=10, cutoff=20, show_spectrum=True, show_error=True):
    noisy = add_noise(clean_signal, snr_db)
    recovered = recover_signal(noisy, cutoff=cutoff, fs=sample_rate)

    recovered_snr = measure_snr(clean_signal, recovered)
    improvement = recovered_snr - snr_db
    scenario = get_scenario_label(snr_db)

    # Styled summary
    display(Markdown(f"""
#Signal and Noise Simulation

**Scenario:** {scenario}
**Input SNR:** {snr_db:.1f} dB
**Recovered SNR:** {recovered_snr:.2f} dB
**Improvement after filtering:** {improvement:.2f} dB
**Filter cutoff:** {cutoff:.1f} Hz
"""))

    # Time-domain plots (we would see how noisy signal and recovered signal gets messy)
    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
    plt.suptitle('Time-domain', fontsize=13, fontweight='bold')

    # 1 - Clean signal
    axes[0].plot(t, clean_signal, linewidth=2, label="Clean signal")
    axes[0].set_title("1. Original Clean Signal")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    # 2 - Noisy signal
    axes[1].plot(t, clean_signal, "--", linewidth=2, label="Original")
    axes[1].plot(t, noisy, alpha=0.7, label=f"Noisy ({snr_db:.1f} dB)")
    if show_error:
        axes[1].fill_between(t, clean_signal, noisy, alpha=0.2, label="Noise error")
    axes[1].set_title("2. Signal After Noise Corruption")
    axes[1].set_ylabel("Amplitude")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    # 3 - Recovery comparison
    axes[2].plot(t, clean_signal, "--", linewidth=2, label="Original")
    axes[2].plot(t, noisy, alpha=0.4, label="Noisy")
    axes[2].plot(t, recovered, linewidth=2, label="Recovered")
    if show_error:
        axes[2].fill_between(t, clean_signal, recovered, alpha=0.15, label="Residual error")
    axes[2].set_title("3. Signal Recovery After Filtering")
    axes[2].set_xlabel("Time (seconds)")
    axes[2].set_ylabel("Amplitude")
    axes[2].grid(alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    # Frequency-domain plots (shows why the time domain plot gets messy)
    if show_spectrum:
        fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
        plt.suptitle('Frequency-domain', fontsize=13, fontweight='bold')
        plot_spectrum(axes[0], clean_signal, sample_rate, "1. Spectrum of Clean Signal")
        plot_spectrum(axes[1], noisy, sample_rate, "2. Spectrum of Noisy Signal")
        plot_spectrum(axes[2], recovered, sample_rate, "3. Spectrum of Recovered Signal")
        axes[2].set_xlim(0, 100)
        plt.tight_layout()
        plt.show()


# Interactive controls where you can adjust the input SNR and cutoff filter frequency
snr_slider = widgets.FloatSlider(
    value=10,
    min=0,
    max=30,
    step=1,
    description='SNR (dB):',
    continuous_update=False
)

cutoff_slider = widgets.FloatSlider(
    value=20,
    min=5,
    max=100,
    step=1,
    description='Cutoff:',
    continuous_update=False
)

spectrum_toggle = widgets.Checkbox(
    value=True,
    description='Show spectrum'
)

error_toggle = widgets.Checkbox(
    value=True,
    description='Show error shading'
)

ui = widgets.VBox([snr_slider, cutoff_slider, spectrum_toggle, error_toggle])
out = widgets.interactive_output(
    simulate,
    {
        'snr_db': snr_slider,
        'cutoff': cutoff_slider,
        'show_spectrum': spectrum_toggle,
        'show_error': error_toggle
    }
)

display(ui, out)
